# Initialization

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#In case, you wish to run code and catboost is not installed then run this cell
!pip install catboost

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings("always")
warnings.filterwarnings("ignore")

from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split, StratifiedShuffleSplit, GridSearchCV
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import accuracy_score, confusion_matrix
from mpl_toolkits.mplot3d import Axes3D
from sklearn import decomposition

In [ ]:
def plot_cm(y_true, y_pred, fname, figsize=(10,10)):
    cm = confusion_matrix(y_true, y_pred, labels=np.unique(y_true))
    cm_sum = np.sum(cm, axis=1, keepdims=True)
    cm_perc = cm / cm_sum.astype(float) * 100
    annot = np.empty_like(cm).astype(str)
    nrows, ncols = cm.shape
    for i in range(nrows):
        for j in range(ncols):
            c = cm[i, j]
            p = cm_perc[i, j]
            if i == j:
                s = cm_sum[i]
                annot[i, j] = '%.1f%%\n%d/%d' % (p, c, s)
            elif c == 0:
                annot[i, j] = ''
            else:
                annot[i, j] = '%.1f%%\n%d' % (p, c)
    cm = pd.DataFrame(cm, index=np.unique(y_true), columns=np.unique(y_true))
    cm.index.name = 'Actual'
    cm.columns.name = 'Predicted'
    fig, ax = plt.subplots(figsize=figsize)
    sns.heatmap(cm, cmap= "YlGnBu", annot=annot, fmt='', ax=ax)
    ax.set_title(fname)
    fig.savefig('/content/drive/MyDrive/CSE578DVGroupProject/'+fname+'.png')

In [ ]:
def modelComparison(data, label, dataset, encoding = 0, scaling = 0, test = 0.2):
  print("With", dataset)
  #Data pre-processing
  cat_cols = [x for x in data.columns.tolist() if data[x].dtype == 'O']
  bool_dict = {}
  cat_dict = {}
  for col in cat_cols:
    temp = list(data[col].unique())
    if (len(temp) == 2):
      bool_dict[col] = temp
    else:
      cat_dict[col] = temp
  print("Boolean-category columns:", bool_dict)
  print("Multi-category columns:", cat_dict)
  #Boolean-Category Columns
  for col in list(bool_dict.keys()):
    for idx, val in enumerate(bool_dict[col]):
      data[col].replace(to_replace = val, value = idx, inplace = True)
  #Multi-Category Columns
  #One-hot encoding
  if (encoding == 0):
    for col in list(cat_dict.keys()):
      data[col] = data[col].apply(lambda x: col+'_'+ x)
      temp = list(data[col].unique())
      for t in temp:
        data[t] = 0
        data.loc[list(data[data[col] == t].index), t] = 1
      data = data.drop(columns = [col])
  #Label encoding
  elif (encoding == 1):
    for col in list(cat_dict.keys()):
      for idx, val in enumerate(cat_dict[col]):
        data[col].replace(to_replace = val, value = idx, inplace = True)
  else:
    print("Input 0 or 1 for encoding")
    return

  #Data Scaling
  #MinMax Scaling
  if (scaling == 0):
    scaler = MinMaxScaler()
    scaledData = scaler.fit_transform(data)
    scaledData = pd.DataFrame(scaledData, columns=data.columns.tolist())
    print("MinMax Scaled Data Shape:", scaledData.shape)
  #Standard Scaling
  elif (scaling == 1):
    scaler = MinMaxScaler()
    scaledData = scaler.fit_transform(data)
    scaledData = pd.DataFrame(scaledData, columns=data.columns.tolist())
    print("Standard Scaled Data Shape:", scaledData.shape)
  #Unscaled
  elif (scaling == 2):
    scaledData = data.copy()
    print("Un-Scaled Data Shape:", scaledData.shape)
  else:
    print("Input 0, 1 or 2 for scaling")
    return

  #Creating Train, Validation and Test splits
  X_temp, X_test, y_temp, y_test = train_test_split(scaledData, label, test_size=test, random_state=19)
  X_train, X_cross, y_train, y_cross = train_test_split(X_temp, y_temp, test_size=test, random_state=19)
  del X_temp, y_temp
  print("Train Data:", X_train.shape, len(y_train))
  print("Validation Data:", X_cross.shape, len(y_cross))
  print("Test Data:", X_test.shape, len(y_test))
  #Initializing cross-validation function
  cv = StratifiedShuffleSplit(n_splits=5, test_size=0.2)
  #12 Models Comparison with various hyper-parameter tuning
  best_models_list, best_train_accuracy, best_test_accuracy, refit_time, misclass = [], [], [], [], []
  
  #Logistic Regression
  clf = LogisticRegression()
  parameters = {'solver': ['newton-cg', 'lbfgs', 'sag', 'saga'], 'max_iter': [1000, 2500, 5000]}
  grid = GridSearchCV(clf, param_grid=parameters, cv=cv, verbose = 1, n_jobs = -1)
  grid.fit(X_train, y_train)
  print("Best Logistic Regression Model:", grid.best_estimator_)
  best_estimator = grid.best_estimator_
  best_models_list.append(best_estimator)
  best_refit = grid.refit_time_
  refit_time.append(best_refit)
  #Fetching predictions on test set for best model
  y_pred = best_estimator.predict(X_cross)
  #Evaluation of best model performance
  best_acc = accuracy_score(y_cross, y_pred)*100
  best_train_accuracy.append(grid.best_score_*100)
  best_test_accuracy.append(best_acc)
  count_misclassified = (y_cross != y_pred).sum()
  misclass.append(count_misclassified)
  plot_cm(y_cross, y_pred, dataset+'_LogReg')
  
  #Stochastic Gradient Descent Classifier
  clf = SGDClassifier()
  parameters = {'learning_rate': ['constant', 'optimal', 'invscaling', 'adaptive'], 'eta0': [0.0001, 0.001, 0.01, 0.1, 1, 10, 100]}
  grid = GridSearchCV(clf, param_grid=parameters, cv=cv, verbose = 1, n_jobs = -1)
  grid.fit(X_train, y_train)
  print("Best Stochastic Gradient Decsent Classifier Model:", grid.best_estimator_)
  best_estimator = grid.best_estimator_
  best_models_list.append(best_estimator)
  best_refit = grid.refit_time_
  refit_time.append(best_refit)
  #Fetching predictions on test set for best model
  y_pred = best_estimator.predict(X_cross)
  #Evaluation of best model performance
  best_acc = accuracy_score(y_cross, y_pred)*100
  best_train_accuracy.append(grid.best_score_*100)
  best_test_accuracy.append(best_acc)
  count_misclassified = (y_cross != y_pred).sum()
  misclass.append(count_misclassified)
  plot_cm(y_cross, y_pred, dataset+'_SGD')
  
  #Decision Tree Classifier
  clf = DecisionTreeClassifier()
  parameters = {'criterion': ['gini', 'entropy'], 'splitter': ['best', 'random']}
  grid = GridSearchCV(clf, param_grid=parameters, cv=cv, verbose = 1, n_jobs = -1)
  grid.fit(X_train, y_train)
  print("Best Decision Tree Classifier Model:", grid.best_estimator_)
  best_estimator = grid.best_estimator_
  best_models_list.append(best_estimator)
  best_refit = grid.refit_time_
  refit_time.append(best_refit)
  #Fetching predictions on test set for best model
  y_pred = best_estimator.predict(X_cross)
  #Evaluation of best model performance
  best_acc = accuracy_score(y_cross, y_pred)*100
  best_train_accuracy.append(grid.best_score_*100)
  best_test_accuracy.append(best_acc)
  count_misclassified = (y_cross != y_pred).sum()
  misclass.append(count_misclassified)
  plot_cm(y_cross, y_pred, dataset+'_Tree')
  
  #Random Forest Classifier
  clf = RandomForestClassifier()
  parameters = {'criterion': ['gini', 'entropy'], 'n_estimators': [25, 50, 100, 200, 500]}
  grid = GridSearchCV(clf, param_grid=parameters, cv=cv, verbose = 1, n_jobs = -1)
  grid.fit(X_train, y_train)
  print("Best Random Forest Classifier Model:", grid.best_estimator_)
  best_estimator = grid.best_estimator_
  best_models_list.append(best_estimator)
  best_refit = grid.refit_time_
  refit_time.append(best_refit)
  #Fetching predictions on test set for best model
  y_pred = best_estimator.predict(X_cross)
  #Evaluation of best model performance
  best_acc = accuracy_score(y_cross, y_pred)*100
  best_train_accuracy.append(grid.best_score_*100)
  best_test_accuracy.append(best_acc)
  count_misclassified = (y_cross != y_pred).sum()
  misclass.append(count_misclassified)
  plot_cm(y_cross, y_pred, dataset+'_RF')
  
  #AdaBoost Classifier
  clf = AdaBoostClassifier()
  parameters = {'n_estimators': [50, 100, 200, 500], 'learning_rate': [0.0001, 0.001, 0.01, 0.1, 1, 10]}
  grid = GridSearchCV(clf, param_grid=parameters, cv=cv, verbose = 1, n_jobs = -1)
  grid.fit(X_train, y_train)
  print("Best AdaBoost Classifier Model:", grid.best_estimator_)
  best_estimator = grid.best_estimator_
  best_models_list.append(best_estimator)
  best_refit = grid.refit_time_
  refit_time.append(best_refit)
  #Fetching predictions on test set for best model
  y_pred = best_estimator.predict(X_cross)
  #Evaluation of best model performance
  best_acc = accuracy_score(y_cross, y_pred)*100
  best_train_accuracy.append(grid.best_score_*100)
  best_test_accuracy.append(best_acc)
  count_misclassified = (y_cross != y_pred).sum()
  misclass.append(count_misclassified)
  plot_cm(y_cross, y_pred, dataset+'_Ada')
  
  #k-Nearest Neighbor Classifier
  clf = KNeighborsClassifier()
  parameters = {'n_neighbors': [1, 5, 10, 20, 50], 'weights': ['uniform', 'distance'], 'algorithm': ['auto', 'ball_tree', 'kd_tree', 'brute']}
  grid = GridSearchCV(clf, param_grid=parameters, cv=cv, verbose = 1, n_jobs = -1)
  grid.fit(X_train, y_train)
  print("Best k-Nearest Neighbor Classifier Model:", grid.best_estimator_)
  best_estimator = grid.best_estimator_
  best_models_list.append(best_estimator)
  best_refit = grid.refit_time_
  refit_time.append(best_refit)
  #Fetching predictions on test set for best model
  y_pred = best_estimator.predict(X_cross)
  #Evaluation of best model performance
  best_acc = accuracy_score(y_cross, y_pred)*100
  best_train_accuracy.append(grid.best_score_*100)
  best_test_accuracy.append(best_acc)
  count_misclassified = (y_cross != y_pred).sum()
  misclass.append(count_misclassified)
  plot_cm(y_cross, y_pred, dataset+'_kNN')
  
  #Linear Discriminant Analysis
  clf = LinearDiscriminantAnalysis()
  parameters = {'solver': ['svd', 'lsqr', 'eigen']}
  grid = GridSearchCV(clf, param_grid=parameters, cv=cv, verbose = 1, n_jobs = -1)
  grid.fit(X_train, y_train)
  print("Best Linear Discriminant Analysis Model:", grid.best_estimator_)
  best_estimator = grid.best_estimator_
  best_models_list.append(best_estimator)
  best_refit = grid.refit_time_
  refit_time.append(best_refit)
  #Fetching predictions on test set for best model
  y_pred = best_estimator.predict(X_cross)
  #Evaluation of best model performance
  best_acc = accuracy_score(y_cross, y_pred)*100
  best_train_accuracy.append(grid.best_score_*100)
  best_test_accuracy.append(best_acc)
  count_misclassified = (y_cross != y_pred).sum()
  misclass.append(count_misclassified)
  plot_cm(y_cross, y_pred, dataset+'_LDA')
  
  #Quadratic Discriminant Analysis
  clf = QuadraticDiscriminantAnalysis()
  parameters = {'reg_param': [0, 0.001, 0.01, 0.1]}
  grid = GridSearchCV(clf, param_grid=parameters, cv=cv, verbose = 1, n_jobs = -1)
  grid.fit(X_train, y_train)
  print("Best Quadratic Discriminant Analysis Model:", grid.best_estimator_)
  best_estimator = grid.best_estimator_
  best_models_list.append(best_estimator)
  best_refit = grid.refit_time_
  refit_time.append(best_refit)
  #Fetching predictions on test set for best model
  y_pred = best_estimator.predict(X_cross)
  #Evaluation of best model performance
  best_acc = accuracy_score(y_cross, y_pred)*100
  best_train_accuracy.append(grid.best_score_*100)
  best_test_accuracy.append(best_acc)
  count_misclassified = (y_cross != y_pred).sum()
  misclass.append(count_misclassified)
  plot_cm(y_cross, y_pred, dataset+'_QDA')
  
  #Gaussian Naive-Bayes Classifier
  clf = GaussianNB()
  parameters = {}
  grid = GridSearchCV(clf, param_grid=parameters, cv=cv, verbose = 1, n_jobs = -1)
  grid.fit(X_train, y_train)
  print("Best Gaussian Naive-Bayes Classifier Model:", grid.best_estimator_)
  best_estimator = grid.best_estimator_
  best_models_list.append(best_estimator)
  best_refit = grid.refit_time_
  refit_time.append(best_refit)
  #Fetching predictions on test set for best model
  y_pred = best_estimator.predict(X_cross)
  #Evaluation of best model performance
  best_acc = accuracy_score(y_cross, y_pred)*100
  best_train_accuracy.append(grid.best_score_*100)
  best_test_accuracy.append(best_acc)
  count_misclassified = (y_cross != y_pred).sum()
  misclass.append(count_misclassified)
  plot_cm(y_cross, y_pred, dataset+'_NB')
  
  #Support Vector Machine
  clf = SVC() 
  parameters = {'C': [1, 10, 100], 'kernel': ['linear', 'rbf']}
  grid = GridSearchCV(clf, param_grid=parameters, cv=cv, verbose = 1, n_jobs = -1)
  grid.fit(X_train, y_train)
  print("Best Support Vector Machine Model:", grid.best_estimator_)
  best_estimator = grid.best_estimator_
  best_models_list.append(best_estimator)
  best_refit = grid.refit_time_
  refit_time.append(best_refit)
  #Fetching predictions on test set for best model
  y_pred = best_estimator.predict(X_cross)
  #Evaluation of best model performance
  best_acc = accuracy_score(y_cross, y_pred)*100
  best_train_accuracy.append(grid.best_score_*100)
  best_test_accuracy.append(best_acc)
  count_misclassified = (y_cross != y_pred).sum()
  misclass.append(count_misclassified)
  plot_cm(y_cross, y_pred, dataset+'_SVM')
  
  #Artificial Neural Network Classifier
  clf = MLPClassifier()
  parameters = {'hidden_layer_sizes': [(5, 2), (4, 4, 2), (20, 2)], 'activation': ['relu'], 
                'solver': ['sgd', 'adam'], 'alpha': [0.0001, 0.01, 0.1, 1]}
  grid = GridSearchCV(clf, param_grid=parameters, cv=cv, verbose = 1, n_jobs = -1)
  grid.fit(X_train, y_train)
  print("Best Artificial Neural Network Classifier Model:", grid.best_estimator_)
  best_estimator = grid.best_estimator_
  best_models_list.append(best_estimator)
  best_refit = grid.refit_time_
  refit_time.append(best_refit)
  #Fetching predictions on test set for best model
  y_pred = best_estimator.predict(X_cross)
  #Evaluation of best model performance
  best_acc = accuracy_score(y_cross, y_pred)*100
  best_train_accuracy.append(grid.best_score_*100)
  best_test_accuracy.append(best_acc)
  count_misclassified = (y_cross != y_pred).sum()
  misclass.append(count_misclassified)
  plot_cm(y_cross, y_pred, dataset+'_ANN')
  
  #Model Performance Comparison 
  model_performance = pd.DataFrame(columns = ['Abbr', 'Model', 'Train Accuracy', 'Validation Accuracy', 'Refit Time', 'Misclassified Samples'])
  model_list = ['Logistic Regression', 'Stochastic Gradient Descent', 'Decision Tree', 'Random Forest', 'AdaBoot', 'k-Nearest Neighbors', 
                'Linear Discriminant Analysis', 'Quadratic Discriminant Analysis', 'Naive-Bayes', 'Support Vector Machine', 'Neural Network']
  
  for i, txt in enumerate(model_list):
      val = {'Abbr': txt, 'Model': best_models_list[i], 'Train Accuracy': best_train_accuracy[i], 
            'Validation Accuracy': best_test_accuracy[i], 'Refit Time': refit_time[i],'Misclassified Samples': misclass[i]}
      model_performance = model_performance.append(val, ignore_index=True)
  model_performance = model_performance.sort_values(by=['Validation Accuracy', 'Train Accuracy', 'Refit Time'], ascending=[False, False, True])
  model_performance = model_performance.reset_index(drop = True)
  return X_test, y_test, model_performance

# Data Pre-Processing

In [ ]:
columns = ['age', 'workclass', 'fnlwgt', 'education', 'education-num', 'marital-status', 'occupation', 
           'relationship', 'race', 'sex', 'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'label']
data = pd.read_csv('/content/drive/MyDrive/CSE578DVGroupProject/adult-all.csv', header = None, na_values='?', names = columns)
data['label'].replace({'<=50K': 0, '>50K': 1}, inplace = True)
data = data.drop(columns = ['education-num'])
data.fillna('NA', inplace = True)
print(data.shape)
data.head()

In [ ]:
data['label'].value_counts()

# Model Training

## Label Encoding

In [ ]:
X = data.drop(columns = ['label'])
y = data['label']
X_test, y_test, model_performance = modelComparison(X, y, 'LE', encoding = 1, scaling = 0, test = 0.2)
model_performance

In [ ]:
model_performance.to_csv('/content/drive/MyDrive/CSE578DVGroupProject/perf.csv', index = False)

## One Hot Encoding

In [ ]:
X = data.drop(columns = ['label'])
y = data['label']
X_test2, y_test2, model_performance2 = modelComparison(X, y, 'OHE', encoding = 0, scaling = 0, test = 0.2)
model_performance2

In [ ]:
model_performance2.to_csv('/content/drive/MyDrive/CSE578DVGroupProject/perf2.csv', index = False)

## CatBoost

In [ ]:
X = data.drop(columns = ['label'])
y = data['label']

catCols = [col for col in X.columns.tolist() if data[col].dtype == 'O']

#Creating Train, Validation and Test splits
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=19)
X_train, X_cross, y_train, y_cross = train_test_split(X_temp, y_temp, test_size=0.2, random_state=19)
del X_temp, y_temp
print("Train Data:", X_train.shape, len(X_train))
print("Validation Data:", X_cross.shape, len(y_cross))
print("Test Data:", X_test.shape, len(y_test))

model = CatBoostClassifier(one_hot_max_size = 100, learning_rate = 0.03, cat_features=catCols)
pool_train = Pool(X_train, y_train, cat_features=catCols, feature_names=list(X_train.columns))
pool_cross = Pool(X_cross, y_cross, cat_features=catCols, feature_names=list(X_cross.columns))
# fit the model with the training data
model.fit(pool_train, eval_set = pool_cross, early_stopping_rounds = 300, verbose=50)

In [ ]:
print('--- CatBoost Results ---')
# predict the target on the train dataset
predict_train = model.predict(X_train, prediction_type = 'Class')
# Accuray Score on train dataset
accuracy_train = accuracy_score(y_train, predict_train)
print('Train Accuracy : ', accuracy_train)

# predict the target on the train dataset
predict_cross = model.predict(X_cross, prediction_type = 'Class')
# Accuray Score on train dataset
accuracy_cross = accuracy_score(y_cross, predict_cross)
print('Validation Accuracy : ', accuracy_cross)

count_misclassified = (y_cross != predict_cross).sum()
print('Number of Misclassfied Samples : ', count_misclassified)

# predict the target on the train dataset
predict_test = model.predict(X_test, prediction_type = 'Class')
# Accuray Score on train dataset
accuracy_test = accuracy_score(y_test, predict_test)
print('CatBoost is the BEST Model.')
print('Test Accuracy : ', accuracy_test)

In [ ]:
model.plot_tree(tree_idx=0, pool=pool_train)

In [ ]:
g = model.plot_tree(tree_idx=0, pool=pool_train)
g.format = 'png'
g.render('/content/drive/MyDrive/CSE578DVGroupProject/CatBoot_Tree', view=True)

In [ ]:
model.save_model('/content/drive/MyDrive/CSE578DVGroupProject/CatBoost_Model', format="cbm")

In [ ]:
plot_cm(y_cross, predict_cross, 'Original'+'_CatBoost')

In [ ]:
columns = model.feature_names_
importance = model.feature_importances_
featureImp = pd.DataFrame()
featureImp['Feature'] = columns
featureImp['Importance'] = importance
featureImp = featureImp.sort_values(by=['Importance'], ascending = False)
featureImp = featureImp.reset_index(drop = True)
featureImp.to_csv('/content/drive/MyDrive/CSE578DVGroupProject/CatBoost_FeatureImp.csv', index = False)
featureImp

# SkyLine Visualization

In [ ]:
#helper function to check whether row1 is better than row2
def betterRow(row1, row2, minimize, maximize):
    better = 0
    for item in maximize:
        if row1[item] < row2[item]:
            return False
        else:
            better += row1[item] > row2[item]
    
    for item in minimize:
        if row1[item] > row2[item]:
            return False
        else:
            better += row1[item] < row2[item]
    
    if better > 0:
        return True
    return False

#helper function to evaluate all rows in data that are not worse than any other row,
# given the constraints to minimize and maximize
def evaluateSkyline(data, minimize, maximize):
    rows = data.to_dict(orient='index')
    answer = set()
    for row1 in rows:
        dominated = False
        for row2 in rows:
            if row1 == row2:
                continue
            elif betterRow(rows[row2], rows[row1], minimize, maximize):
                dominated = True
                break
        if not dominated:
            answer.add(row1)
    return pd.Series(data.index.isin(answer), index=data.index)

In [ ]:
tempData1 = pd.read_csv('/content/drive/MyDrive/CSE578DVGroupProject/perf.csv')
tempData1['Dataset'] = 1
tempData2 = pd.read_csv('/content/drive/MyDrive/CSE578DVGroupProject/perf2.csv')
tempData2['Dataset'] = 2
performanceData = pd.concat([tempData1, tempData2]).reset_index(drop=True)
meanAcc = performanceData['Validation Accuracy'].mean()
meanFit = performanceData['Refit Time'].mean()
print(performanceData.shape)
performanceData = performanceData[performanceData['Validation Accuracy'] >= meanAcc]
performanceData = performanceData[performanceData['Refit Time'] <= meanFit]
performanceData = performanceData.reset_index(drop = True)
print(performanceData.shape)
performanceData

In [ ]:
skyline_row_idx = evaluateSkyline(performanceData, ['Refit Time'], ['Validation Accuracy'])
performanceData[skyline_row_idx]

In [ ]:
colors = skyline_row_idx.map({True: 'skyblue', False: 'grey'})
plt.figure(figsize=(20, 10))
plt.scatter(performanceData['Refit Time'], performanceData['Validation Accuracy'], c=colors, edgecolors = 'black')
plt.title('Skyline for Model Performancce')
plt.xlabel('Refit Time (in seconds)')
plt.ylabel('Validation Accuracy')
plt.legend(['SkyLine Models'], loc = 'upper left')

#Annotate
count=0
for idx, row in performanceData.iterrows():
    if (skyline_row_idx[idx]):
        if (count == 0):
            x, y = 0, -0.15
        else:
            x, y = -0.05, 0.10
        txt = str(row['Abbr'])+str(", D=")+str(row['Dataset'])
        plt.annotate(txt, (row['Refit Time']+x, row['Validation Accuracy']+y))
        count+=1
plt.savefig('/content/drive/MyDrive/CSE578DVGroupProject/skyline.png')
plt.show()

# Principal Component Analysis

## Pre Process

In [ ]:
def preprocess(data, scaling = 0):
  #Data pre-processing
  cat_cols = [x for x in data.columns.tolist() if data[x].dtype == 'O']
  bool_dict = {}
  cat_dict = {}
  for col in cat_cols:
    temp = list(data[col].unique())
    if (len(temp) == 2):
      bool_dict[col] = temp
    else:
      cat_dict[col] = temp
  # print("Boolean-category columns:", bool_dict)
  # print("Multi-category columns:", cat_dict)
  #Boolean-Category Columns
  for col in list(bool_dict.keys()):
    for idx, val in enumerate(bool_dict[col]):
      data[col].replace(to_replace = val, value = idx, inplace = True)
  #Multi-Category Columns
  for col in list(cat_dict.keys()):
    for idx, val in enumerate(cat_dict[col]):
      data[col].replace(to_replace = val, value = idx, inplace = True)
  if (scaling == 0):
    print("Label-Encoded Data Shape:", data.shape)
    return data
  
  scaler = MinMaxScaler()
  scaledData = scaler.fit_transform(data)
  scaledData = pd.DataFrame(scaledData, columns=data.columns.tolist())
  print("MinMax Scaled Data Shape:", scaledData.shape)
  return scaledData

## 3D Visualization

In [ ]:
X = data.drop(columns = ['label'])
preX0 = preprocess(X)
X = data.drop(columns = ['label'])
preX1 = preprocess(X, scaling = 1)
y = data['label']
y = np.choose(y, [1, 0]).astype(float)
print(preX0.shape, preX1.shape, y.shape)

In [ ]:
fig = plt.figure(1, figsize=(15, 15))
plt.clf()
ax = Axes3D(fig, rect=[0, 0, .95, 1], elev=48, azim=134)

plt.cla()
pca = decomposition.PCA(n_components=3)
pca.fit(preX0)
preX0 = pca.transform(preX0)

for name, label in [('> $50K', 1), ('<= $50K', 0)]:
    ax.text3D(preX0[y == label, 0].mean(), preX0[y == label, 1].mean() + 1.5, preX0[y == label, 2].mean(), name,
              horizontalalignment='center', bbox=dict(alpha=.5, edgecolor='w', facecolor='w'))
# Reorder the labels to have colors matching the cluster results
ax.scatter(preX0[:, 0], preX0[:, 1], preX0[:, 2], c=y, cmap=plt.cm.nipy_spectral, edgecolor='k')

ax.w_xaxis.set_ticklabels([])
ax.w_yaxis.set_ticklabels([])
ax.w_zaxis.set_ticklabels([])
plt.title('PCA without Scaling')
plt.savefig('/content/drive/MyDrive/CSE578DVGroupProject/pca3dwoscale.png')
plt.tight_layout()
plt.show()

In [ ]:
fig = plt.figure(1, figsize=(10, 10))
plt.clf()
ax = Axes3D(fig, rect=[0, 0, .95, 1], elev=48, azim=134)

plt.cla()
pca = decomposition.PCA(n_components=3)
pca.fit(preX1)
preX1 = pca.transform(preX1)

for name, label in [('> $50K', 1), ('<= $50K', 0)]:
    ax.text3D(preX1[y == label, 0].mean(), preX1[y == label, 1].mean() + 1.5, preX1[y == label, 2].mean(), name,
              horizontalalignment='center', bbox=dict(alpha=.5, edgecolor='w', facecolor='w'))
# Reorder the labels to have colors matching the cluster results
ax.scatter(preX1[:, 0], preX1[:, 1], preX1[:, 2], c=y, cmap=plt.cm.nipy_spectral, edgecolor='k')

ax.w_xaxis.set_ticklabels([])
ax.w_yaxis.set_ticklabels([])
ax.w_zaxis.set_ticklabels([])
plt.title('PCA with Scaling')
plt.savefig('/content/drive/MyDrive/CSE578DVGroupProject/pca3dwscale.png')
plt.tight_layout()
plt.show()

## Analysis

In [ ]:
X = data.drop(columns = ['label'])
preX = preprocess(X, scaling = 1)
y = data['label']

In [ ]:
pca1 = decomposition.PCA()
pca1.fit(preX)
eigenvalues = pca1.explained_variance_
print(eigenvalues)

In [ ]:
columns = ['age', 'workclass', 'fnlwgt', 'education', 'marital-status', 'occupation', 
           'relationship', 'race', 'sex', 'capital-gain', 'capital-loss', 'hours-per-week', 'native-country']
featureImp1 = pd.DataFrame()
featureImp1['Feature'] = columns
featureImp1['EigenValue'] = eigenvalues
featureImp1 = featureImp1.sort_values(by=['EigenValue'], ascending = False)
featureImp1 = featureImp1.reset_index(drop = True)
featureImp1.to_csv('/content/drive/MyDrive/CSE578DVGroupProject/PCA_FeatureImp.csv', index = False)
featureImp1